## 1. SVM regression — K-fold with NAVG & KEEP_LAST_PERCENT


## 2. Imports & Global Config

In [2]:
# =========================
# Core Python
# =========================
import os
import sys
import json
import csv
import logging
from collections import defaultdict
from typing import List, Tuple, Dict, Any

# =========================
# Numerical & Data Handling
# =========================
import numpy as np
import pandas as pd

# =========================
# Machine Learning
# =========================
from sklearn.svm import SVR
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    accuracy_score,
    matthews_corrcoef
)

# =========================
# Statistics
# =========================
from scipy.stats import pearsonr

# =========================
# Model Persistence
# =========================
import joblib

# =========================
# Reproducibility
# =========================
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 3. Load Experimental & MD Data, Merge on Sequence

Assumes:
- `exp_data_all.csv` contains at least: `sequence`, `bind_avg`  
- `rawdat.csv` contains MD features plus `sequence` and `run`  


In [3]:
# ID and label columns
id_col    = "sequence"
label_col = "bind_avg"

# --- Experimental data ---
df_exp = pd.read_csv("../Data/exp_data_all.csv")
ref_data = df_exp[[id_col, label_col]].copy()
print("Experimental data:", ref_data.shape)
display(ref_data.head())

# --- MD / feature data ---
usecols = [
    "sequence", "run",
    "VDWAALS", "EEL", "EGB", "ESURF",
    "HB Energy", "Hydrophobic Energy", "Pi-Pi Energy",
    "Delta_Entropy"
]

Experimental data: (168, 2)


,sequence,bind_avg
0,GAGGAAGCAGCCCTCGCCCCTGTCGGTGGAAAGAAG,-0.758634
1,GCAGCCGAGGCGGAGAGAGAGAGAGGACAGCTTACG,-1.003319
2,ATCTGATCAAAACAACGAATTCCAAAACAAAGTAAT,-0.800322
3,CCAATATTCCTTTGTGAGACCCTCCACAAATGCTAA,-0.941242
4,GAGGACGCGAACCGGCACGCTGCGCCTTTAAGGAGT,-0.684116


In [4]:
df_md = pd.read_csv("../Data/rawdat.csv", usecols=usecols)
feature_data = df_md.copy()
print("Feature data:", feature_data.shape)
display(feature_data.head())


Feature data: (272160, 10)


,sequence,run,VDWAALS,EEL,EGB,ESURF,HB Energy,Hydrophobic Energy,Pi-Pi Energy,Delta_Entropy
0,GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC,9,-252.110,-1886.830,1841.253,-36.482,-1.940432,-165.447020,-1.655191e-03,-26.046553
1,GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC,9,-238.510,-1881.424,1835.847,-36.023,-2.003962,-155.422935,-4.708262e-02,-24.150637
2,GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC,9,-246.721,-1895.687,1851.589,-35.802,-2.269901,-142.386371,-5.901517e-29,-24.329875
3,GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC,9,-235.671,-1857.573,1814.002,-34.799,-2.838678,-147.918585,-3.236084e-07,-23.615145
4,GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC,9,-230.214,-1897.268,1847.934,-34.391,-2.810414,-151.012478,-1.784784e-05,-23.698348


In [5]:

# --- Merge on sequence ---
df_merged = pd.merge(feature_data, ref_data, on=id_col, how="inner")
print("Merged data:", df_merged.shape)
display(df_merged.head())

# Shuffle merged rows
df_merged = df_merged.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
print("Merged (shuffled):", df_merged.shape)

Merged data: (272160, 11)


,sequence,run,VDWAALS,EEL,EGB,ESURF,HB Energy,Hydrophobic Energy,Pi-Pi Energy,Delta_Entropy,bind_avg
0,GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC,9,-252.110,-1886.830,1841.253,-36.482,-1.940432,-165.447020,-1.655191e-03,-26.046553,1.531336
1,GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC,9,-238.510,-1881.424,1835.847,-36.023,-2.003962,-155.422935,-4.708262e-02,-24.150637,1.531336
2,GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC,9,-246.721,-1895.687,1851.589,-35.802,-2.269901,-142.386371,-5.901517e-29,-24.329875,1.531336
3,GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC,9,-235.671,-1857.573,1814.002,-34.799,-2.838678,-147.918585,-3.236084e-07,-23.615145,1.531336
4,GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC,9,-230.214,-1897.268,1847.934,-34.391,-2.810414,-151.012478,-1.784784e-05,-23.698348,1.531336


Merged (shuffled): (272160, 11)


## 4. Initial Train/Test Split by Sequence (80/20)

We split at the **sequence level** so that sequences in test are not seen in training.


In [6]:
test_percentage = 0.20

unique_seqs = df_merged[id_col].unique()
np.random.seed(RANDOM_STATE)
np.random.shuffle(unique_seqs)

n_train = int((1.0 - test_percentage) * len(unique_seqs))

train_seqs = unique_seqs[:n_train]
test_seqs  = unique_seqs[n_train:]

df_train = df_merged[df_merged[id_col].isin(train_seqs)].copy()
df_test  = df_merged[df_merged[id_col].isin(test_seqs)].copy()

print("Train shape:", df_train.shape)
print("Test  shape:", df_test.shape)

# Save initial split (optional, for traceability)
df_train.to_csv("reg_trn_final.csv", index=False)
df_test.to_csv("reg_tst_preprocess.csv", index=False)

Train shape: (217080, 11)
Test  shape: (55080, 11)


In [13]:
if "run" in df_train.columns:
    df_train.drop(columns=["run"], inplace=True, errors="ignore")
if "run" in df_test.columns:
    df_test.drop(columns=["run"], inplace=True, errors="ignore")

print(df_train.shape, df_test.shape) # 8 features, 1 sequence, 1 label

(217080, 10) (55080, 10)


In [15]:
# Save
train_file = f"reg_trn_final.csv"
test_file  = f"reg_tst_preprocess.csv"

df_train.to_csv(train_file, index=False)
df_test.to_csv(test_file, index=False)

## 5. KEEP_LAST_PERCENT = 50

For each sequence, keep only the last 50% of rows (e.g., later frames / runs).


In [19]:
KEEP_LAST_PERCENT = 100  # from hyperparameter search

df_sorted = df_train.copy()

group_sizes = df_sorted.groupby(id_col)[id_col].transform("size")
cumcount    = df_sorted.groupby(id_col).cumcount()

n_keep = (group_sizes * (KEEP_LAST_PERCENT / 100.0)).astype(int)
n_keep = n_keep.mask(n_keep < 1, 1)  # at least 1 row if fraction > 0

mask = cumcount >= (group_sizes - n_keep)
df_train = df_sorted[mask].reset_index(drop=True)

print("After KEEP_LAST_PERCENT:", df_train.shape)
display(df_train.head())
print(group_sizes.unique())
# print(cumcount)

After KEEP_LAST_PERCENT: (217080, 10)


,sequence,VDWAALS,EEL,EGB,ESURF,HB Energy,Hydrophobic Energy,Pi-Pi Energy,Delta_Entropy,bind_avg
0,CGGCTTTTTCTTGAACACGTGGAATATACTAGCGCT,-207.264,-1958.570,1908.590,-34.442,-3.478696,-140.468722,-5.839134,-22.871906,1.976320
1,TTAGAAAAATAGTTTAAAATCTAGAGTTAATTAACC,-197.506,-1830.441,1787.958,-29.223,-2.514019,-120.507243,-4.043271,-18.798981,0.002694
2,CCAGCTCTCCACCGCCGCGTGCGCCTGCAGACGCTC,-207.553,-1891.794,1845.707,-31.909,-13.343306,-138.470452,-3.535617,-22.011054,0.153350
3,CCCCCAGCGCTCCGGCACGCGCCGGGAGACCTCCGG,-201.484,-1923.635,1874.505,-31.059,-4.628190,-138.103395,-0.006414,-22.886912,-0.483023
4,TCCGCCTCCGTCCCCCACGTTGCGTTCTGGGAGTTG,-203.541,-1924.910,1872.754,-32.963,-3.595719,-144.264637,-4.272515,-23.900001,0.033983


[1620]


## 6. NAVG = 20 — Average Numeric Features Per Sequence

In [20]:
NAVG = 20  # from hyperparameter search

def average_features_for_sequence(
    df: pd.DataFrame,
    navg: int,
    id_col: str,
    label_col: str,
    random_state: int
) -> pd.DataFrame:
    """
    Shuffle the sequence's rows, chunk into size navg, and average numeric features.
    Label = label from the first row of each chunk.
    """
    results = []
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    feature_cols = [c for c in numeric_cols if c not in [id_col, label_col]]
    df_shuffled = df.sample(frac=1.0, random_state=random_state).reset_index(drop=True)
    n_chunks = len(df_shuffled) // navg
    if n_chunks < 1:
        return pd.DataFrame(columns=df.columns)
    for i in range(n_chunks):
        chunk = df_shuffled.iloc[i * navg : (i + 1) * navg]
        row_dict = {col: chunk[col].mean() for col in feature_cols}
        row_dict[label_col] = chunk[label_col].iloc[0]
        row_dict[id_col] = chunk[id_col].iloc[0]
        results.append(row_dict)
    df_out = pd.DataFrame(results)
    col_order = [id_col] + sorted([c for c in df_out.columns if c != id_col])
    return df_out[col_order]

def average_features_for_mutants(
    df: pd.DataFrame,
    navg: int,
    id_col: str,
    label_col: str,
    random_state: int
) -> pd.DataFrame:
    """
    Apply the above averaging function per sequence group, then concat.
    """
    all_chunks = []
    for seq, group in df.groupby(id_col):
        chunk_df = average_features_for_sequence(group, navg, id_col, label_col, random_state)
        all_chunks.append(chunk_df)
    if not all_chunks:
        return pd.DataFrame(columns=df.columns)
    return pd.concat(all_chunks, ignore_index=True)


In [22]:
df_train_avg = average_features_for_mutants(
    df_train,
    navg=NAVG,
    id_col=id_col,
    label_col=label_col,
    random_state=RANDOM_STATE
)

print("After NAVG averaging:", df_train_avg.shape)
display(df_train_avg.head())

After NAVG averaging: (10854, 10)


,sequence,Delta_Entropy,EEL,EGB,ESURF,HB Energy,Hydrophobic Energy,Pi-Pi Energy,VDWAALS,bind_avg
0,AACCACTCGACTGACCTCGTGGTCAAATTCCTTACT,-23.422286,-1889.25845,1842.90880,-32.95365,-13.547972,-145.302473,-2.042028,-216.04365,1.172119
1,AACCACTCGACTGACCTCGTGGTCAAATTCCTTACT,-22.274451,-1885.08835,1839.20195,-31.94215,-14.572665,-139.682198,-2.487052,-211.11960,1.172119
2,AACCACTCGACTGACCTCGTGGTCAAATTCCTTACT,-22.675960,-1889.28055,1841.86925,-31.79340,-14.453291,-139.509590,-2.803277,-208.69625,1.172119
3,AACCACTCGACTGACCTCGTGGTCAAATTCCTTACT,-22.172318,-1878.70740,1832.68840,-31.13805,-13.350216,-137.357618,-1.799540,-205.71785,1.172119
4,AACCACTCGACTGACCTCGTGGTCAAATTCCTTACT,-22.951100,-1895.92670,1848.62760,-32.89245,-13.314758,-142.652436,-2.941463,-215.72575,1.172119


## 7. Compute Mean/Std, Save, and Standardize

Compute Z-score stats on training (averaged) data and apply them to get
a standardized training table.


In [23]:
def compute_mean_std(
    df: pd.DataFrame,
    model_type: str,
    id_col: str,
    label_col: str
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Compute mean/std for numeric columns (excluding ID and label).
    """
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if id_col in numeric_cols:
        numeric_cols.remove(id_col)
    if label_col in numeric_cols:
        numeric_cols.remove(label_col)
    means = [(c, df[c].mean()) for c in numeric_cols]
    stds  = [(c, df[c].std())  for c in numeric_cols]
    return (
        pd.DataFrame(means, columns=["colname", "mean"]),
        pd.DataFrame(stds, columns=["colname", "std"])
    )

def save_mean_std(mean_df: pd.DataFrame, std_df: pd.DataFrame, filename: str) -> None:
    merged = pd.merge(mean_df, std_df, on="colname")
    merged.to_csv(filename, index=False)

def load_mean_std(filename: str) -> pd.DataFrame:
    if not os.path.isfile(filename):
        logging.error(f"Mean/Std file not found: {filename}")
        sys.exit(1)
    df_stats = pd.read_csv(filename)
    needed_cols = {"colname", "mean", "std"}
    if not needed_cols.issubset(df_stats.columns):
        logging.error(f"Mean/Std file missing columns. Found: {df_stats.columns.tolist()}")
        sys.exit(1)
    return df_stats

def apply_standardization(
    df: pd.DataFrame,
    stats_df: pd.DataFrame,
    model_type: str,
    id_col: str,
    label_col: str
) -> pd.DataFrame:
    df_std = df.copy()
    means = dict(zip(stats_df["colname"], stats_df["mean"]))
    stds  = dict(zip(stats_df["colname"], stats_df["std"]))
    for col in df_std.columns:
        if col in [id_col, label_col]:
            continue
        if col in means and col in stds:
            mu  = means[col]
            sigma = stds[col]
            if sigma == 0 or np.isnan(sigma):
                df_std[col] = df_std[col] - mu
            else:
                df_std[col] = (df_std[col] - mu) / sigma
    return df_std

In [24]:

# Compute and save stats
mean_df, std_df = compute_mean_std(df_train_avg, 'reg', id_col, label_col)
save_mean_std(mean_df, std_df, "reg_train_stats.csv")

# Apply standardization
stats_dict = load_mean_std("reg_train_stats.csv")
df_train_std = apply_standardization(df_train_avg, stats_dict, 'reg', id_col, label_col)

print("Standardized training data:", df_train_std.shape)
display(df_train_std.head())

Standardized training data: (10854, 10)


,sequence,Delta_Entropy,EEL,EGB,ESURF,HB Energy,Hydrophobic Energy,Pi-Pi Energy,VDWAALS,bind_avg
0,AACCACTCGACTGACCTCGTGGTCAAATTCCTTACT,-1.583414,0.696472,-0.591948,-0.928603,-0.985700,-0.836708,1.087607,-0.975745,1.172119
1,AACCACTCGACTGACCTCGTGGTCAAATTCCTTACT,-0.288769,1.046877,-0.931115,-0.066938,-1.171832,-0.129708,0.605042,-0.516780,1.172119
2,AACCACTCGACTGACCTCGTGGTCAAATTCCTTACT,-0.741632,0.694614,-0.687064,0.059777,-1.150148,-0.107995,0.262142,-0.290903,1.172119
3,AACCACTCGACTGACCTCGTGGTCAAATTCCTTACT,-0.173573,1.583055,-1.527087,0.618049,-0.949779,0.162711,1.350550,-0.013290,1.172119
4,AACCACTCGACTGACCTCGTGGTCAAATTCCTTACT,-1.051962,0.136152,-0.068694,-0.876469,-0.943338,-0.503348,0.112299,-0.946114,1.172119


In [25]:
# Shuffle once more
df_train_std = df_train_std.sample(frac=1.0, random_state=42).reset_index(drop=True)
print(df_train_std)


                                   sequence  Delta_Entropy       EEL  \
0      CTTTAGATTTTTTTCTTACCTTGTTCTAGCAATTAG       1.704638 -1.546593   
1      TCCAATAAAAAAGAATAAAAAAGAAATACCTGGGTC       3.420110  0.627262   
2      TAAGGGGTGACCCAGCCGCTGCAGAGCCAGGGAAGG       0.933260  0.386921   
3      TCACTCGGGCACTTCCGCGTGGAATAGGAGGCGCCA      -0.405895  1.035235   
4      TCTTGGTCTGAAGAGCACATGGCATGTCAAGGTCAC       0.203061  0.093869   
...                                     ...            ...       ...   
10849  GACACTAAGCTTCTTCCCAGGTTTTCAGATTCCGAA      -0.077063  0.746754   
10850  CTGTCCTCTCTCGCCCACGCTGCCTGGGAGGCGCGC      -0.516873  1.219294   
10851  CTTTAGATTTTTTTCTTACCTTGTTCTAGCAATTAG       2.601976  0.031445   
10852  AGAGGGGAATCTTTCCACGTGCACCCAGTTCTTTTT      -0.573048  0.411848   
10853  GTAGTGCATAGCTACCTCATGGGCTACAGCCAGCTC      -0.135575  0.075656   

            EGB     ESURF  HB Energy  Hydrophobic Energy  Pi-Pi Energy  \
0      1.505347  2.103878  -1.101297            2.491162     

## 8. Repeated Group K-Fold (on Sequence IDs)

We now create:
- `reg_trn_{repeat}_{fold}.csv`  
- `reg_val_{repeat}_{fold}.csv`  
for cross-validation.


In [26]:
num_repeats = 3
kfold       = 5

groups = df_train_std[id_col]
print(groups)


0        CTTTAGATTTTTTTCTTACCTTGTTCTAGCAATTAG
1        TCCAATAAAAAAGAATAAAAAAGAAATACCTGGGTC
2        TAAGGGGTGACCCAGCCGCTGCAGAGCCAGGGAAGG
3        TCACTCGGGCACTTCCGCGTGGAATAGGAGGCGCCA
4        TCTTGGTCTGAAGAGCACATGGCATGTCAAGGTCAC
                         ...                 
10849    GACACTAAGCTTCTTCCCAGGTTTTCAGATTCCGAA
10850    CTGTCCTCTCTCGCCCACGCTGCCTGGGAGGCGCGC
10851    CTTTAGATTTTTTTCTTACCTTGTTCTAGCAATTAG
10852    AGAGGGGAATCTTTCCACGTGCACCCAGTTCTTTTT
10853    GTAGTGCATAGCTACCTCATGGGCTACAGCCAGCTC
Name: sequence, Length: 10854, dtype: object


In [ ]:
for repeat_idx in range(num_repeats):
    repeat_seed = random_state + 100 * repeat_idx
    np.random.seed(repeat_seed)

    kf = GroupKFold(n_splits=kfold)
    split_iter = kf.split(df_train_std, groups=groups)

    fold_counter = 0
    for trn_idx, val_idx in split_iter:
        df_fold_trn = df_train_std.iloc[trn_idx].copy()
        df_fold_val = df_train_std.iloc[val_idx].copy()

        # Ensure ID is first column
        col_order = [id_col] + [c for c in df_fold_trn.columns if c != id_col]
        df_fold_trn = df_fold_trn[col_order]
        df_fold_val = df_fold_val[col_order]

        fold_train_csv = f"reg_trn_{repeat_idx}_{fold_counter}.csv"
        fold_val_csv   = f"reg_val_{repeat_idx}_{fold_counter}.csv"

        df_fold_trn.to_csv(fold_train_csv, index=False)
        df_fold_val.to_csv(fold_val_csv, index=False)

        print("Wrote:", fold_train_csv, "|", fold_val_csv)
        fold_counter += 1

In [12]:
# ---------- Literature hyperparams ----------
KEEP_LAST_PERCENT = 100   # keep all
NAVG = 20                 # average every 20 rows within each sequence
KERNEL = "rbf"
C = 1.81
GAMMA = 0.004
EPSILON = 0.1            # if literature didn’t specify, keep a standard value

# ---------- Optional: regression→classification threshold metrics ----------
DATA_SCALE = "log"        # if your labels are log ΔΔG, threshold at 0.0
THRESH = 0.0 if DATA_SCALE == "log" else 1.0